In [1]:
import numpy as np
from scipy import stats
from math import ceil
from pathlib import Path
import matplotlib.pyplot as plt
from contact_plots import get_data_from_files, get_cdf_of_data, do_chi_square_test
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder)

In [3]:
cut_off_distances = ['1.0', '1.5', '2.0']

data_names_base = (
    'customer_customer_weight_over_contacts',
    'customer_customer_weight_over_agents',
    'customer_customer_contacts_over_agents',
    'staff_customer_weight_over_contacts',
    'staff_customer_weight_over_agents',
    'staff_customer_contacts_over_agents',
    'customer_staff_weight_over_agents',
    'customer_staff_contacts_over_agents',
)

parts_2_data_name = {}
for cut_off_distance in cut_off_distances:
    data_name_dict = {}
    for data_name in data_names_base:
        parts = data_name.split('_')
        interaction_type = '_'.join(parts[:2])
        data_type = '_'.join(parts[2:])
        if interaction_type not in data_name_dict:
            data_name_dict[interaction_type] = {}
            
        data_name_dict[interaction_type][data_type] = f'{cut_off_distance}_{data_name}'
    
    parts_2_data_name[cut_off_distance] = data_name_dict
    
measure_2_label = {
    'nn_nts_bvd':'Base',
    'n_nts_bvd':'Nbrh',
    'nn_ts_bvd':'Ts',
    'nn_nts_svd':'Short',
    'nn_nts_lvd':'Long'
}

base_measure_name = 'nn_nts_bvd'

In [4]:
%matplotlib wx

infra_name_all = 'base_case'
interaction_type_all = 'customer_customer'
cut_off_dist = '1.5'

plots = [
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'weight_over_contacts'},
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'weight_over_agents'},
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'contacts_over_agents'}
]

row_count = 1 
col_count = len(plots)
fig, axes = plt.subplots(row_count, col_count)
fig.canvas.manager.set_window_title(f'{infra_name_all} - {interaction_type_all}')
axes = axes.flatten()

for ax_nr, plot_info in enumerate(plots):
    ax = axes[ax_nr]
    infra_name = plot_info['infra_name']
    data_name = parts_2_data_name[cut_off_dist][plot_info['interaction_type']][plot_info['data_type']]
    
    box_data = []
    labels = []
    base_data = None
    
    for measure_name, measure_label in measure_2_label.items():
        box_data.append(infra_and_measure_name_2_data[infra_name][measure_name][data_name])
        if measure_name == base_measure_name:
            base_data = box_data[-1]

        labels.append(measure_label)
    
    ax.axhline(np.mean(base_data), linewidth=1, color='b', linestyle='-')
    ax.boxplot(box_data, vert=True, labels=labels, showmeans=True, meanline=True, meanprops={'linewidth':2, 'color':'r'})
    
    ax.grid()
    ax.set_title(plot_info['data_type'])

plt.draw()

In [5]:
%matplotlib wx

infra_name_all = 'base_case'
interaction_type_all = 'customer_staff'
cut_off_dist = '1.5'

stacked_bar_cats = [1,2,3]
stacked_bar_data_type = 'contacts_over_agents'

plots = [
    {'infra_name':infra_name_all, 'interaction_type': 'staff_customer', 'data_type':'weight_over_contacts'},
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'weight_over_agents'},
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'contacts_over_agents'}
]

row_count = 1 
col_count = len(plots)
fig, axes = plt.subplots(row_count, col_count)
fig.canvas.manager.set_window_title(f'{infra_name_all} - {interaction_type_all}')
axes = axes.flatten()

for ax_nr, plot_info in enumerate(plots):
    ax = axes[ax_nr]
    infra_name = plot_info['infra_name']
    data_name = parts_2_data_name[cut_off_dist][plot_info['interaction_type']][plot_info['data_type']]
    
    box_data = []
    labels = []
    base_data = None
    bar_data = {category:[] for category in stacked_bar_cats}
    
    for measure_name, measure_label in measure_2_label.items():
        if plot_info['data_type'] == stacked_bar_data_type:
            contact_data = infra_and_measure_name_2_data[infra_name][measure_name][data_name]
            contact_data_count = len(contact_data)
            for category in stacked_bar_cats:
                bar_data[category].append(np.sum(contact_data == category)/contact_data_count)
        else:
            box_data.append(infra_and_measure_name_2_data[infra_name][measure_name][data_name])
            if measure_name == base_measure_name:
                base_data = box_data[-1]

        labels.append(measure_label)
    
    if plot_info['data_type'] == stacked_bar_data_type:
        bottoms = None
                
        for category in stacked_bar_cats:
            if bottoms is None:
                ax.bar(labels, bar_data[category], 0.5, label=f'{category}')
                bottoms = bar_data[category]
            else:
                ax.bar(labels, bar_data[category], 0.5, label=f'{category}', bottom=bottoms)
                bottoms = [bottoms[ii] + bar_data[category][ii] for ii in range(len(labels))]
        ax.legend()        
    else:
        ax.axhline(np.mean(base_data), linewidth=1, color='b', linestyle='-')
        ax.boxplot(box_data, vert=True, labels=labels, showmeans=True, meanline=True, meanprops={'linewidth':2, 'color':'r'})
    
    ax.grid()
    ax.set_title(plot_info['data_type'])

plt.draw()

In [6]:
%matplotlib wx

infra_name_all = 'base_case'
interaction_type_all = 'staff_customer'
cut_off_dist = '1.5'

plots = [
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'weight_over_contacts'},
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'weight_over_agents'},
    {'infra_name':infra_name_all, 'interaction_type': interaction_type_all, 'data_type':'contacts_over_agents'}
]

row_count = 1 
col_count = len(plots)
fig, axes = plt.subplots(row_count, col_count)
fig.canvas.manager.set_window_title(f'{infra_name_all} - {interaction_type_all}')
axes = axes.flatten()

for ax_nr, plot_info in enumerate(plots):
    ax = axes[ax_nr]
    infra_name = plot_info['infra_name']
    data_name = parts_2_data_name[cut_off_dist][plot_info['interaction_type']][plot_info['data_type']]
    
    box_data = []
    labels = []
    base_data = None
    
    for measure_name, measure_label in measure_2_label.items():
        box_data.append(infra_and_measure_name_2_data[infra_name][measure_name][data_name])
        if measure_name == base_measure_name:
            base_data = box_data[-1]

        labels.append(measure_label)
    
    ax.axhline(np.mean(base_data), linewidth=1, color='b', linestyle='-')
    ax.boxplot(box_data, vert=True, labels=labels, showmeans=True, meanline=True, meanprops={'linewidth':2, 'color':'r'})
    
    ax.grid()
    ax.set_title(plot_info['data_type'])

plt.draw()

In [7]:
plt.show()